# Model D: combined geometric and low-light augmentation

Model D completes the augmentation ablation. It combines Model B's geometric transformations with Model C's randomized low-light corruption while leaving the data split, EfficientNetB0 backbone, classification head, optimizer, callbacks, and clean test set unchanged.

This creates a simple 2 × 2 design: Model A uses neither augmentation family, Model B uses geometry only, Model C uses low light only, and Model D uses both. Because Model B reduced clean performance, Model D is still necessary to test whether geometry interacts differently with illumination augmentation.

In [ ]:
from pathlib import Path
import json
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

SEED = 42
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print(f"Python: {sys.version.split()[0]}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")

## Experiment settings

The geometric ranges are copied from Model B. The low-light probability and parameter ranges are copied from Model C. None of these values are adjusted using the results of Model D.

In [ ]:
CLASS_NAMES = ['angry', 'happy', 'sad']
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/Emotions Dataset')
LOCAL_DATASET_DIR = Path('/content/emotions_dataset')
PROJECT_OUTPUT_DIR = Path('/content/drive/MyDrive/CNN-Robustness-Low-Light-Analysis/outputs')
OUTPUT_DIR = PROJECT_OUTPUT_DIR / 'model_d'

RESULT_PATHS = {
    'Model A': PROJECT_OUTPUT_DIR / 'model_a' / 'results_summary.json',
    'Model B': PROJECT_OUTPUT_DIR / 'model_b' / 'results_summary.json',
    'Model C': PROJECT_OUTPUT_DIR / 'model_c' / 'results_summary.json',
}

CONFIG = {
    'image_size': (224, 224),
    'batch_size': 32,
    'validation_split': 0.20,
    'epochs': 30,
    'learning_rate': 1e-3,
    'dropout_rate': 0.30,
    'dense_units': 256,
    'rotation_degrees': 10,
    'translation_fraction': 0.10,
    'zoom_fraction': 0.10,
    'low_light_probability': 0.75,
    'gamma_range': (1.10, 1.80),
    'exposure_range': (0.60, 0.95),
    'noise_stddev_range': (0.00, 0.03),
}

CONFIG

## Prepare the dataset and previous results

As in Models A–C, the original training folder supplies the training and validation subsets, and the original test folder is untouched until evaluation. The previous JSON summaries are loaded directly for comparison.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

required_directories = [
    DRIVE_DATASET_DIR / split / class_name
    for split in ['train', 'test']
    for class_name in CLASS_NAMES
]
missing_directories = [path for path in required_directories if not path.is_dir()]

if missing_directories:
    missing_text = '\n'.join(str(path) for path in missing_directories)
    raise FileNotFoundError(f"The following dataset folders were not found:\n{missing_text}")

if not LOCAL_DATASET_DIR.exists():
    print('Copying the dataset from Drive to the Colab runtime...')
    shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
    print('Copy complete.')
else:
    print('Using the dataset already copied to the Colab runtime.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_DIR = LOCAL_DATASET_DIR / 'train'
TEST_DIR = LOCAL_DATASET_DIR / 'test'

previous_results = {}
for model_name, result_path in RESULT_PATHS.items():
    if not result_path.is_file():
        raise FileNotFoundError(
            f"{model_name} results were not found at {result_path}. "
            f"Run the {model_name} notebook first."
        )
    with open(result_path) as file:
        previous_results[model_name] = json.load(file)

for model_name, result in previous_results.items():
    print(
        f"{model_name}: accuracy={result['test_accuracy']:.4f}, "
        f"macro-F1={result['test_macro_f1']:.4f}"
    )

In [ ]:
common_dataset_options = {
    'image_size': CONFIG['image_size'],
    'batch_size': CONFIG['batch_size'],
    'label_mode': 'categorical',
    'class_names': CLASS_NAMES,
}

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=CONFIG['validation_split'],
    subset='training',
    seed=SEED,
    shuffle=True,
    **common_dataset_options,
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=CONFIG['validation_split'],
    subset='validation',
    seed=SEED,
    shuffle=True,
    **common_dataset_options,
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    **common_dataset_options,
)

test_file_paths = list(test_dataset.file_paths)
AUTOTUNE = tf.data.AUTOTUNE

## Define the two augmentation components

The illumination transform temporarily works on normalized 0–1 pixels and then returns 0–255 images for EfficientNet. It is mapped only onto the training dataset. The geometric block sits inside the model and is therefore active only during training. In the combined pipeline, low-light corruption is followed by the geometric block.

In [ ]:
def apply_random_low_light(images):
    images = tf.cast(images, tf.float32) / 255.0
    images = tf.clip_by_value(images, 0.0, 1.0)

    parameter_shape = [tf.shape(images)[0], 1, 1, 1]
    apply_mask = tf.random.uniform(
        parameter_shape, seed=SEED
    ) < CONFIG['low_light_probability']

    gamma = tf.random.uniform(
        parameter_shape,
        minval=CONFIG['gamma_range'][0],
        maxval=CONFIG['gamma_range'][1],
        seed=SEED + 1,
    )
    exposure = tf.random.uniform(
        parameter_shape,
        minval=CONFIG['exposure_range'][0],
        maxval=CONFIG['exposure_range'][1],
        seed=SEED + 2,
    )
    noise_stddev = tf.random.uniform(
        parameter_shape,
        minval=CONFIG['noise_stddev_range'][0],
        maxval=CONFIG['noise_stddev_range'][1],
        seed=SEED + 3,
    )

    darkened_images = exposure * tf.pow(images, gamma)
    noise = tf.random.normal(tf.shape(images), seed=SEED + 4) * noise_stddev
    corrupted_images = tf.clip_by_value(darkened_images + noise, 0.0, 1.0)
    output_images = tf.where(apply_mask, corrupted_images, images)

    return output_images * 255.0


def augment_training_batch(images, labels):
    return apply_random_low_light(images), labels


combined_train_dataset = train_dataset.map(
    augment_training_batch,
    # Stateful random operations use one worker for a reproducible sequence.
    num_parallel_calls=1,
    deterministic=True,
).prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [ ]:
geometric_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(
            factor=CONFIG['rotation_degrees'] / 360,
            fill_mode='reflect',
            seed=SEED + 1,
        ),
        layers.RandomTranslation(
            height_factor=CONFIG['translation_fraction'],
            width_factor=CONFIG['translation_fraction'],
            fill_mode='reflect',
            seed=SEED + 2,
        ),
        layers.RandomZoom(
            height_factor=(-CONFIG['zoom_fraction'], CONFIG['zoom_fraction']),
            width_factor=(-CONFIG['zoom_fraction'], CONFIG['zoom_fraction']),
            fill_mode='reflect',
            seed=SEED + 3,
        ),
    ],
    name='geometric_augmentation',
)

## Inspect the combined transformation

The three rows separate the two operations: the middle row shows only Model C's illumination change, and the bottom row then adds Model B's geometry. This check is performed before training so obviously destructive settings can be detected without looking at test performance.

In [ ]:
sample_images, sample_labels = next(iter(train_dataset))
sample_low_light = apply_random_low_light(sample_images[:6])
sample_combined = geometric_augmentation(sample_low_light, training=True)

fig, axes = plt.subplots(3, 6, figsize=(15, 7))
for index in range(6):
    label_index = int(tf.argmax(sample_labels[index]))
    images_to_show = [
        sample_images[index],
        sample_low_light[index],
        sample_combined[index],
    ]

    for row, image in enumerate(images_to_show):
        clipped_image = tf.clip_by_value(image, 0, 255)
        axes[row, index].imshow(clipped_image.numpy().astype('uint8'))
        axes[row, index].axis('off')

    axes[0, index].set_title(CLASS_NAMES[label_index])

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Low light', fontsize=12)
axes[2, 0].set_ylabel('Combined', fontsize=12)
fig.suptitle('Combined augmentation sanity check')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'augmentation_preview.png', dpi=200, bbox_inches='tight')
plt.show()

## Define Model D

Model D adds the same geometric block used by Model B before the otherwise unchanged EfficientNetB0 classifier. The low-light transform is supplied by `combined_train_dataset`, so the saved model still accepts ordinary 0–255 images.

In [ ]:
tf.keras.utils.set_random_seed(SEED)

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=CONFIG['image_size'] + (3,),
)
base_model.trainable = False

inputs = tf.keras.Input(shape=CONFIG['image_size'] + (3,), name='image')
augmented_inputs = geometric_augmentation(inputs)
features = base_model(augmented_inputs, training=False)
features = layers.GlobalAveragePooling2D(name='global_average_pooling')(features)
features = layers.Dropout(CONFIG['dropout_rate'], name='dropout')(features)
features = layers.Dense(CONFIG['dense_units'], activation='relu', name='classifier_dense')(features)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax', name='emotion')(features)

model = tf.keras.Model(inputs, outputs, name='model_d_combined_augmentation')
model.summary()

trainable_parameters = sum(np.prod(variable.shape) for variable in model.trainable_weights)
print(f"Trainable parameters: {trainable_parameters:,}")

## Train with the same selection protocol

The best checkpoint is selected by clean validation loss, using the same callbacks as Models A–C. The original test set is not passed to `fit()`.

In [ ]:
checkpoint_path = OUTPUT_DIR / 'model_d_combined_augmentation.keras'
training_log_path = OUTPUT_DIR / 'training_log.csv'

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top_2_accuracy'),
    ],
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        mode='min',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(training_log_path)),
]

In [ ]:
history = model.fit(
    combined_train_dataset,
    validation_data=validation_dataset,
    epochs=CONFIG['epochs'],
    callbacks=callbacks,
    verbose=2,
)

## Evaluate the saved checkpoint on clean test images

This measures Model D's clean-test performance. Robustness conclusions are postponed until the same fixed low-light conditions are applied to all four saved checkpoints.

In [ ]:
best_model = tf.keras.models.load_model(str(checkpoint_path))
test_metrics = best_model.evaluate(test_dataset, return_dict=True, verbose=1)

print('\nClean test metrics')
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
probabilities = best_model.predict(test_dataset, verbose=1)
predicted_labels = np.argmax(probabilities, axis=1)
true_labels = np.concatenate([
    np.argmax(batch_labels.numpy(), axis=1)
    for _, batch_labels in test_dataset
])

report = classification_report(
    true_labels,
    predicted_labels,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
    output_dict=True,
)

print(classification_report(
    true_labels,
    predicted_labels,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

predictions = pd.DataFrame({
    'file': test_file_paths,
    'true_label': [CLASS_NAMES[index] for index in true_labels],
    'predicted_label': [CLASS_NAMES[index] for index in predicted_labels],
})
for class_index, class_name in enumerate(CLASS_NAMES):
    predictions[f'prob_{class_name}'] = probabilities[:, class_index]

predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

## Inspect the training behavior and class-level errors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs_ran = range(1, len(history.history['loss']) + 1)

axes[0].plot(epochs_ran, history.history['accuracy'], label='Training')
axes[0].plot(epochs_ran, history.history['val_accuracy'], label='Validation')
axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(epochs_ran, history.history['loss'], label='Training')
axes[1].plot(epochs_ran, history.history['val_loss'], label='Validation')
axes[1].set(title='Cross-entropy loss', xlabel='Epoch', ylabel='Loss')
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.suptitle('Model D training history')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

matrix = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title('Model D: clean test confusion matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

## Compare all four models on the clean test set

In [ ]:
model_d_accuracy = float(test_metrics['accuracy'])
model_d_macro_f1 = float(report['macro avg']['f1-score'])

comparison = pd.DataFrame({
    'Model': [
        'Model A: none',
        'Model B: geometry',
        'Model C: low light',
        'Model D: combined',
    ],
    'Accuracy': [
        previous_results['Model A']['test_accuracy'],
        previous_results['Model B']['test_accuracy'],
        previous_results['Model C']['test_accuracy'],
        model_d_accuracy,
    ],
    'Macro-F1': [
        previous_results['Model A']['test_macro_f1'],
        previous_results['Model B']['test_macro_f1'],
        previous_results['Model C']['test_macro_f1'],
        model_d_macro_f1,
    ],
})
display(comparison.style.format({'Accuracy': '{:.4f}', 'Macro-F1': '{:.4f}'}))

comparison_plot = comparison.set_index('Model').plot.bar(
    figsize=(10, 5),
    ylim=(0, 1),
    rot=0,
    color=['#4C72B0', '#55A868'],
)
comparison_plot.set_ylabel('Score')
comparison_plot.set_title('Clean-test comparison')
comparison_plot.grid(axis='y', alpha=0.25)
comparison_plot.figure.tight_layout()
comparison_plot.figure.savefig(
    OUTPUT_DIR / 'comparison_with_models_a_b_c.png',
    dpi=200,
    bbox_inches='tight',
)
plt.show()

In [ ]:
history_to_save = {
    name: [float(value) for value in values]
    for name, values in history.history.items()
}

with open(OUTPUT_DIR / 'training_history.json', 'w') as file:
    json.dump(history_to_save, file, indent=2)

model_a_accuracy = float(previous_results['Model A']['test_accuracy'])
model_a_macro_f1 = float(previous_results['Model A']['test_macro_f1'])
model_c_accuracy = float(previous_results['Model C']['test_accuracy'])
model_c_macro_f1 = float(previous_results['Model C']['test_macro_f1'])
best_epoch = int(np.argmin(history.history['val_loss']) + 1)

results_summary = {
    'experiment': 'Model D - combined geometric and low-light augmentation',
    'seed': SEED,
    'tensorflow_version': tf.__version__,
    'image_size': list(CONFIG['image_size']),
    'batch_size': CONFIG['batch_size'],
    'validation_split': CONFIG['validation_split'],
    'augmentation': {
        'order': ['low_light', 'geometric'],
        'low_light': {
            'formula': 'clip(exposure * (image / 255) ** gamma + noise, 0, 1) * 255',
            'application_probability': CONFIG['low_light_probability'],
            'gamma_range': list(CONFIG['gamma_range']),
            'exposure_range': list(CONFIG['exposure_range']),
            'noise_stddev_range_normalized': list(CONFIG['noise_stddev_range']),
        },
        'geometric': {
            'horizontal_flip': True,
            'rotation_degrees': CONFIG['rotation_degrees'],
            'translation_fraction': CONFIG['translation_fraction'],
            'zoom_fraction': CONFIG['zoom_fraction'],
        },
        'validation_and_test_augmented': False,
    },
    'epochs_completed': len(history.history['loss']),
    'best_epoch_by_validation_loss': best_epoch,
    'best_validation_loss': float(min(history.history['val_loss'])),
    'best_validation_accuracy': float(max(history.history['val_accuracy'])),
    'test_condition': 'clean',
    'test_loss': float(test_metrics['loss']),
    'test_accuracy': model_d_accuracy,
    'test_top_2_accuracy': float(test_metrics['top_2_accuracy']),
    'test_macro_f1': model_d_macro_f1,
    'test_weighted_f1': float(report['weighted avg']['f1-score']),
    'accuracy_change_from_model_a': model_d_accuracy - model_a_accuracy,
    'macro_f1_change_from_model_a': model_d_macro_f1 - model_a_macro_f1,
    'accuracy_change_from_model_c': model_d_accuracy - model_c_accuracy,
    'macro_f1_change_from_model_c': model_d_macro_f1 - model_c_macro_f1,
    'classification_report': report,
}

with open(OUTPUT_DIR / 'results_summary.json', 'w') as file:
    json.dump(results_summary, file, indent=2)

print(json.dumps({
    'best_epoch': results_summary['best_epoch_by_validation_loss'],
    'test_condition': results_summary['test_condition'],
    'test_accuracy': results_summary['test_accuracy'],
    'test_macro_f1': results_summary['test_macro_f1'],
    'accuracy_change_from_model_a': results_summary['accuracy_change_from_model_a'],
    'macro_f1_change_from_model_a': results_summary['macro_f1_change_from_model_a'],
    'accuracy_change_from_model_c': results_summary['accuracy_change_from_model_c'],
    'macro_f1_change_from_model_c': results_summary['macro_f1_change_from_model_c'],
}, indent=2))
print(f"\nSaved all Model D outputs to: {OUTPUT_DIR}")

## Next experiment

Keep the complete `model_d` output folder. With Models A–D trained, the next notebook will freeze their checkpoints and evaluate every model on the same clean, mild, moderate, and severe low-light test sets. That evaluation—not the clean comparison alone—will answer the robustness question.